# Tratamento da coleta (Octoparse)

O Octoparse exporta a página de cada trabalho em **formato longo**: uma linha por campo, com o rótulo (`Field1`, ex.: *"Resumo em português"*) e o valor (`Field3`). Cada trabalho começa na linha do **DOI**.

Este notebook:
1. separa os trabalhos e transforma os rótulos em colunas (uma linha por trabalho);
2. limpa os textos e padroniza as colunas conforme o dicionário de dados (`DATA/README.md`);
3. remove os dados pessoais que não entram na análise (autor, banca, e-mail, nome do PDF);
4. confere se a coleta está completa e salva `DATA/teses_usp_pln.csv`, usado pelo notebook de análise.

## 1. Configuração

In [ ]:
from pathlib import Path
import re
import pandas as pd

ARQUIVO_OCTOPARSE = Path('../DATA/Biblioteca_de_Teses_-_Home1.xlsx')  # exportação do Octoparse (.xlsx ou .csv)
ARQUIVO_SAIDA = Path('../DATA/teses_usp_pln.csv')
TOTAL_ESPERADO = 127          # total de resultados da busca no site
DATA_COLETA = '2026-10-09'    # data em que a coleta foi feita

## 2. Leitura e separação dos trabalhos

In [ ]:
if ARQUIVO_OCTOPARSE.suffix == '.csv':
    bruto = pd.read_csv(ARQUIVO_OCTOPARSE)
else:
    bruto = pd.read_excel(ARQUIVO_OCTOPARSE)

ROTULO, VALOR, VALOR_LINK = 'Field1', 'Field3', 'Field2_text'

def limpar(texto):
    """Junta quebras de linha e espaços repetidos."""
    return ' '.join(str(texto).split()) if pd.notna(texto) else None

bruto['trabalho'] = bruto[ROTULO].eq('DOI').cumsum()   # um novo trabalho a cada linha de DOI
bruto['valor'] = bruto[VALOR].fillna(bruto[VALOR_LINK]).map(limpar)

campos = bruto.dropna(subset=[ROTULO])
campos = campos[~campos[ROTULO].str.lower().str.endswith('.pdf')]   # nomes de arquivo PDF

largo = campos.pivot_table(index='trabalho', columns=ROTULO, values='valor', aggfunc='first')
print(f'{len(bruto)} linhas na exportação → {len(largo)} trabalhos')
largo.notna().sum().sort_values(ascending=False).rename('preenchidos').to_frame()

## 3. Padronização das colunas

In [ ]:
def coluna(nome):
    return largo[nome] if nome in largo else pd.Series(None, index=largo.index, dtype='object')

def sem_parenteses(nome):
    """'Gomi, Edson Satoshi ( )' → 'Gomi, Edson Satoshi'"""
    return re.sub(r'\s*\(.*?\)\s*$', '', nome).strip() if isinstance(nome, str) else nome

df = pd.DataFrame({
    'doi': coluna('DOI'),
    'titulo': coluna('Título em português').fillna(coluna('Título em inglês')),
    'titulo_en': coluna('Título em inglês'),
    'nivel': coluna('Documento'),
    'unidade': coluna('Unidade da USP'),
    'area_concentracao': coluna('Área de Concentração'),
    'orientador': coluna('Orientador').map(sem_parenteses),
    'data_defesa': pd.to_datetime(coluna('Data de Defesa'), errors='coerce'),
    'palavras_chave': coluna('Palavras-chave em português'),
    'palavras_chave_en': coluna('Palavras-chave em inglês'),
    'resumo_pt': coluna('Resumo em português'),
    'resumo_en': coluna('Resumo em inglês'),
}).reset_index(drop=True)

df['ano'] = df['data_defesa'].dt.year.astype('Int64')
df['id_trabalho'] = df['doi'].str.extract(r'(tde-[\d-]+)', expand=False)
df['url'] = 'https://doi.org/' + df['doi']
df['data_coleta'] = DATA_COLETA

# O DOI da USP indica o nível (D = dissertação, T = tese) e o código da unidade: 10.11606/D.3.2007...
partes = df['doi'].str.extract(r'10\.11606/(?P<tipo>[DT])\.(?P<cod_unidade>\d+)\.')
df['nivel'] = df['nivel'].fillna(partes['tipo'].map({'D': 'Dissertação de Mestrado', 'T': 'Tese de Doutorado'}))

# Unidade: completa os vazios com a unidade de outros trabalhos que têm o mesmo código no DOI
df['cod_unidade'] = partes['cod_unidade']
mapa = df.dropna(subset=['unidade']).drop_duplicates('cod_unidade').set_index('cod_unidade')['unidade']
df['unidade'] = df['unidade'].fillna(df['cod_unidade'].map(mapa))

df.head(3).T

## 4. Conferência da coleta

In [ ]:
print(f'Trabalhos: {len(df)} de {TOTAL_ESPERADO} esperados')
if len(df) < TOTAL_ESPERADO:
    print(f'⚠️ Faltam {TOTAL_ESPERADO - len(df)} trabalhos. Verifique se todas as páginas de resultados foram coletadas.')
print('DOIs duplicados:', df['doi'].duplicated().sum())

vazios = df.drop(columns=['data_coleta', 'url', 'id_trabalho']).isna().sum()
vazios[vazios > 0].sort_values(ascending=False).rename('vazios').to_frame()

**Recorte temático:** a busca foi feita por *"processamento de linguagem natural"*. A célula abaixo mostra quantos trabalhos têm o termo (em português ou inglês) no título ou nas palavras-chave. Leia os que não têm antes de decidir se ficam na análise.

In [ ]:
texto_busca = (df['titulo'].fillna('') + ' ' + df['titulo_en'].fillna('') + ' ' +
               df['palavras_chave'].fillna('') + ' ' + df['palavras_chave_en'].fillna('')).str.lower()
df['tem_termo_pln'] = texto_busca.str.contains('processamento de linguagem natural|natural language processing')
print(df['tem_termo_pln'].value_counts().rename({True: 'com o termo', False: 'sem o termo'}).to_string())
df.loc[~df['tem_termo_pln'], ['ano', 'titulo']]

## 5. Exportação

In [ ]:
COLUNAS = ['id_trabalho', 'doi', 'titulo', 'titulo_en', 'nivel', 'unidade', 'area_concentracao',
           'orientador', 'ano', 'data_defesa', 'palavras_chave', 'palavras_chave_en',
           'resumo_pt', 'resumo_en', 'tem_termo_pln', 'url', 'data_coleta']
df[COLUNAS].sort_values('ano').to_csv(ARQUIVO_SAIDA, index=False, encoding='utf-8')
print('Salvo em', ARQUIVO_SAIDA.resolve())